###

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-30")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
#movies_df = spark.read.parquet(f"{silver_folder_path}/movies")
movies_df = spark.read.table("movie_silver.movies")\
                        .filter(f"file_date = '{v_file_date}'")

#country_df = spark.read.parquet(f"{silver_folder_path}/countries")
country_df = spark.read.table("movie_silver.countries")

#prduct_country_df = spark.read.parquet(f"{silver_folder_path}/productions_countries")
prduct_country_df = spark.read.table("movie_silver.productions_countries")\
                        .filter(f"file_date = '{v_file_date}'")


In [0]:
movies_join_df = movies_df.join(prduct_country_df,
                                movies_df.movie_id == prduct_country_df.movie_id, "inner")\
                            .join(country_df,
                                  prduct_country_df.country_id == country_df.country_id, "inner")\
                            .select(
                                    "year_release_date",
                                    "budget",
                                    "revenue",
                                    "country_name"
                                    )
                            

In [0]:
from pyspark.sql.functions import sum, dense_rank, desc, lit
from pyspark.sql.window import Window

In [0]:
window = Window.partitionBy("year_release_date").orderBy(desc("total_budget"), desc("total_revenue"))
final_movies_df = movies_join_df\
                .filter("year_release_date >= 2015")\
                .groupBy("year_release_date", "country_name")\
                .agg(
                    sum("budget").alias("total_budget"),
                    sum("revenue").alias("total_revenue")
                )\
                .withColumn("dense_rank", dense_rank().over(window))\
                .withColumn("created_date", lit(v_file_date))


In [0]:
#overwrite_partition("movie_gold", "results_group_movie_country", "created_date", v_file_date)

In [0]:
#final_movies_df.write.mode("overwrite").parquet(f"{gold_folder_path}/results_group_movie_country")

#final_movies_df.write.mode("append").partitionBy("created_date").format("delta").saveAsTable("movie_gold.results_group_movie_country")

condition_merge = 'tgt.year_release_date = src.year_release_date AND tgt.country_name = src.country_name AND tgt.created_date = src.created_date'

incremental_merge("movie_gold", "results_group_movie_country", final_movies_df, condition_merge, "created_date")

In [0]:
%sql
SELECT * FROM movie_gold.results_group_movie_country

year_release_date,country_name,total_budget,total_revenue,dense_rank,created_date
2015,United States of America,5.143775E9,1.8450384353E10,1,2024-12-30
2015,United Kingdom,6.5152236E8,1.894996027E9,2,2024-12-30
2015,Canada,3.245E8,1.334394558E9,3,2024-12-30
2015,Germany,3.095E8,9.5835032E8,4,2024-12-30
2015,China,2.8E8,8.8867852E8,5,2024-12-30
2015,Australia,2.13E8,6.79034882E8,6,2024-12-30
2015,Japan,1.9E8,1.50624936E9,7,2024-12-30
2015,France,1.51E8,2.06495048E8,8,2024-12-30
2015,Hong Kong,1.35E8,5.32950503E8,9,2024-12-30
2015,Taiwan,1.35E8,5.32950503E8,9,2024-12-30


In [0]:
%sql
SELECT created_date, count(1)
FROM movie_gold.results_group_movie_country
GROUP BY created_date;

created_date,count(1)
2024-12-23,2
2024-12-30,45
